<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_2_1_1_LogReg_Intro_w_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to logistic regression with titanic

Author: Brad Sheese

---

# Introduction to Classification - Logistic Regression

This notebook introduces 'classification' through logistic regression. This is a form of supervised learning because we use a dataset where the labels (the outcome we want to predict) are already known. By the end of this exercise, you should be able to train a model to predict a discrete category based on input data.

Before starting, you should be comfortable with simple linear regression. If you need a refresher, please review those exercises first.

---
## Goals of Classification with logistic regression
To 'classify' means to take a specific case in our dataset and assign it to a discrete category. For example, imagine a dataset of fruits with physical dimensions (length, width, height) and labels (apple or banana). We want to determine if these dimensions can predict the fruit's classification. Perhaps length is a reliable way to differentiate an apple from a banana.

This differs from linear regression, where we predict continuous outcomes (like price or temperature). In logistic regression, we want to discover which parameters—often called features—help us differentiate between categories.

---

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.metrics import precision_score, recall_score, accuracy_score, precision_recall_curve, average_precision_score

sns.set_theme(style="whitegrid")

## Load and Explore the Data

Before building any model, we should understand our data. The Titanic dataset contains passenger information: class, sex, age, family details, and fare. Let's load it and see what we're working with.

In [ ]:
url = 'https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv'
df = pd.read_csv(url)

df.columns = df.columns.str.lower().str.replace(' aboard', '', regex=False)
df.info()

### Exploratory Data Analysis

Let's visualize how survival relates to key features. The cultural memory of Titanic is "women and children first" — does the data support this?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(data=df, x='sex', hue='survived', ax=axes[0])
axes[0].set_title('Survival by Sex')
axes[0].legend(['Died', 'Survived'])

sns.boxplot(data=df, x='survived', y='age', ax=axes[1])
axes[1].set_title('Age Distribution by Survival')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Died', 'Survived'])

pd.crosstab(df['pclass'], df['survived']).plot(kind='bar', ax=axes[2])
axes[2].set_title('Survival by Passenger Class')
axes[2].legend(['Died', 'Survived'])
axes[2].set_xlabel('Passenger Class')

plt.tight_layout()
plt.show()

print("Survival rate by sex:")
print(df.groupby('sex')['survived'].mean())
print("\nSurvival rate by class:")
print(df.groupby('pclass')['survived'].mean())

**Observations:**
- **Sex** is a strong predictor: women survived at a much higher rate than men
- **Age** shows a slight trend — younger passengers may have had better survival odds
- **Passenger class** matters: 1st class passengers survived at a much higher rate than 3rd class

Note the **class imbalance**: about 62% of passengers died, 38% survived. We'll come back to this.

## Feature Engineering

Machine learning models require numerical input. We need to convert categorical text (like sex) into numbers and handle skewed distributions (like fare).

In [ ]:
# Binary indicators for family presence
df['sibspouse'] = (df['siblings/spouses'] > 0).astype(int)
df['parentchild'] = (df['parents/children'] > 0).astype(int)

# Encode sex: male=0, female=1
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

# Log-transform fare to handle skew (add 1 to avoid log(0))
df['fare_log'] = np.log1p(df['fare'])

# One-hot encode passenger class with drop_first=True
# This avoids the dummy variable trap (perfect multicollinearity)
# pclass_1 is dropped (reference); pclass_2 and pclass_3 are the dummy columns
df = pd.get_dummies(df, columns=['pclass'], prefix='pclass', drop_first=True)

sns.histplot(df['fare_log'], kde=True)
plt.title('Log-transformed Fare Distribution')
plt.show()

print("Columns after encoding:")
print(df.columns.tolist())

### Train-Test Split

We split the data into a training set (to teach the model) and a test set (to evaluate its performance on unseen data). We use **stratification** to preserve the class ratio in both splits.

In [ ]:
# With drop_first=True, pclass_1 is the reference. pclass_2 and pclass_3 are the dummy columns.
features = ['age', 'sibspouse', 'parentchild', 'fare_log', 'sex', 'pclass_2', 'pclass_3']
X = df[features]
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples ({y_train.mean():.1%} survived)")
print(f"Test set:     {X_test.shape[0]} samples ({y_test.mean():.1%} survived)")

### Building the Pipeline

We use a `Pipeline` to chain preprocessing and modeling together. This ensures that transformations like scaling are applied consistently and prevents data leakage from the test set into training.

**Why scale features?** Logistic regression uses gradient descent to find optimal coefficients. When features are on different scales (e.g., age is 0-80, fare_log is 0-5), the optimization path becomes inefficient. StandardScaler centers each feature at mean=0 with unit variance, helping the solver converge faster and more stably. Note: tree-based models don't need scaling because they make threshold-based splits, not distance calculations.

In [ ]:
numeric_features = ['age', 'fare_log']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ],
    remainder='passthrough'
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

pipeline.fit(X_train, y_train)

train_acc = pipeline.score(X_train, y_train)
test_acc = pipeline.score(X_test, y_test)
print(f"Training Accuracy: {train_acc:.3f}")
print(f"Testing Accuracy:  {test_acc:.3f}")

### Cross-Validation: Is Our Model Stable?

The accuracy on our single test set is useful, but it can be influenced by how the data happened to be split. **K-Fold Cross-Validation** provides a more robust estimate of model performance by splitting the training data into $K$ subsets (folds). The model is trained $K$ times, each time using a different fold as the validation set and the remaining folds for training.

This helps us understand if our model is consistent or if its performance varies significantly depending on the data it sees.

In [ ]:
# Perform 5-fold cross-validation on the training set
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')

print(f"Cross-Validation Accuracy Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")

### How Good Is 76%? Establishing a Baseline

An accuracy of 76% might sound decent, but we need to compare against simple baselines to know if our model is actually adding value.

In [ ]:
# Baseline 1: Always predict the majority class (died)
baseline_all_died = max(y_test.mean(), 1 - y_test.mean())

# Baseline 2: Simple rule - all women survive, all men die
X_test_gender = X_test.copy()
baseline_gender_pred = (X_test_gender['sex'] == 1).astype(int)
baseline_gender_acc = (baseline_gender_pred == y_test).mean()

print(f"Our model accuracy:              {test_acc:.3f}")
print(f"Baseline (predict all died):     {baseline_all_died:.3f}")
print(f"Baseline (women survive rule):   {baseline_gender_acc:.3f}")
print(f"\nOur model improves over the 'all died' baseline by {test_acc - baseline_all_died:.1%}")

### Interpreting the Model: Odds Ratios

The coefficients in a logistic regression model represent log-odds. To make them more intuitive, we exponentiate them to get **odds ratios**. An odds ratio greater than 1 means the feature increases the odds of survival; less than 1 means it decreases them.

In [ ]:
# Get feature names from the preprocessor to ensure they match the coefficients
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
# Clean up the names (remove the 'num__' and 'remainder__' prefixes)
feature_names = [name.split('__')[-1] for name in feature_names]

coefficients = pipeline.named_steps['classifier'].coef_[0]
betas = pd.DataFrame({'feature': feature_names, 'beta': coefficients})
betas['odds_ratio'] = np.exp(betas['beta'])
betas.sort_values(by='odds_ratio', ascending=False)

**Key takeaways:**

- **Sex dominates**: being female increases the odds of survival by roughly 12x, all else equal
- **Age** decreases survival odds: older passengers were less likely to survive
- **Pclass_1** is the reference category (dropped by `drop_first=True`). Both `pclass_2` and `pclass_3` have odds ratios below 1, meaning 2nd and 3rd class passengers had lower survival odds than 1st class
- Higher **fares** increased survival odds
- Having **family members aboard** (siblings, spouses, parents, or children) generally correlated with lower survival odds for individuals, all else equal

Note: odds ratios tell us **direction and magnitude** of effect, but not statistical significance. For confidence intervals and p-values, we'd use `statsmodels`.

### Understanding Predicted Probabilities

Logistic regression produces probabilities, not just hard labels. Let's examine the distribution of predicted probabilities for passengers who actually died vs. survived.

In [ ]:
y_proba = pipeline.predict_proba(X_test)[:, 1]

plt.figure(figsize=(8, 4))
for survived in [0, 1]:
    sns.kdeplot(y_proba[y_test == survived],
                label=f"{'Died' if survived == 0 else 'Survived'}",
                fill=True, alpha=0.4)
plt.axvline(0.5, color='gray', linestyle='--', label='Default threshold (0.5)')
plt.xlabel('Predicted Probability of Survival')
plt.ylabel('Density')
plt.title('Predicted Probability Distributions by Actual Outcome')
plt.legend()
plt.show()

### Model Evaluation: Classification Report

Accuracy alone doesn't tell the full story. The classification report gives us per-class metrics:
- **Precision**: When the model predicts "Survived", how often is it correct?
- **Recall**: Of all actual survivors, what fraction did the model catch?
- **F1-score**: Harmonic mean of precision and recall

In [ ]:
y_pred = pipeline.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

print("\nConfusion Matrix:")
disp = ConfusionMatrixDisplay.from_estimator(
    pipeline, X_test, y_test, cmap='Blues',
    display_labels=['Deceased', 'Survived']
)
plt.grid(False)
plt.show()

**Reading the report:**

- For "Died" (class 0): precision=0.79, recall=0.85 — we correctly identified 85% of actual deaths
- For "Survived" (class 1): precision=0.73, recall=0.64 — we caught 64% of actual survivors
- The model is better at predicting death than survival, partly because the class is larger (imbalance)
- Look at the confusion matrix: the off-diagonal cells show how many survivors we missed (false negatives) and how many deaths we falsely predicted as survival (false positives)

**Which error is worse on the Titanic?** Falsely telling someone their loved one survived when they died (FN), or falsely reporting a death (FP)? This is a business context question — there's no single right answer.

### ROC Curve and AUC

The **ROC curve** plots the True Positive Rate (recall) against the False Positive Rate at every possible threshold. **AUC** (Area Under the Curve) measures the model's ability to rank positive instances above negative ones. AUC=0.5 is random; AUC=1.0 is perfect.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic')
plt.legend()
plt.grid(True)
plt.show()

print(f"AUC: {roc_auc:.3f}")
print(f"Interpretation: there is a {roc_auc:.1%} chance the model will rank a random survivor above a random non-survivor.")

### Precision-Recall Curve

While the ROC curve is excellent for balanced datasets, the **Precision-Recall (PR) curve** is often more informative when dealing with class imbalance. It focuses on the performance of the positive class (survivors).

A PR curve plots precision vs. recall for different thresholds. The **Average Precision (AP)** summarizes the curve as the weighted mean of precisions achieved at each threshold.

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, color='blue', lw=2, label=f'PR curve (AP = {avg_precision:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(True)
plt.show()

print(f"Average Precision: {avg_precision:.3f}")

### Threshold Tuning

The default decision threshold is 0.5, but we can adjust it depending on our priorities. Lowering the threshold catches more survivors (higher recall) but increases false positives (lower precision). Raising it does the opposite.

Let's explore how different thresholds affect the precision-recall tradeoff.

In [ ]:
thresholds_to_test = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []

for t in thresholds_to_test:
    y_pred_t = (y_proba >= t).astype(int)
    results.append({
        'Threshold': t,
        'Accuracy': accuracy_score(y_test, y_pred_t),
        'Precision': precision_score(y_test, y_pred_t),
        'Recall': recall_score(y_test, y_pred_t)
    })

pd.DataFrame(results).set_index('Threshold').round(3)

**Observations:**
- At **t=0.3**: recall is 0.81 (we catch most survivors), but precision drops to 0.65 (more false alarms)
- At **t=0.4**: accuracy peaks at 0.78 — the best balance for this dataset with equal error costs
- At **t=0.7**: precision reaches 0.93, but recall plummets to 0.44 (we miss over half the survivors)
- No single threshold is "best" — it depends on the cost of each error type

In a real application, you'd choose the threshold that minimizes expected cost, using business context to assign dollar values to false positives and false negatives.

## When Does Logistic Regression Work Well?

Now that we've built and evaluated a model, let's discuss the conditions where logistic regression performs best:

1. **Binary or ordinal outcome** — the target has two distinct categories (it can be extended to multiclass via softmax)

2. **Low multicollinearity** — features shouldn't be too strongly correlated with each other. Our one-hot encoding with `drop_first=True` helps avoid this. A common rule: correlations between predictors should stay below ~0.80

3. **Independent observations** — each row should represent a unique individual (not repeated measures). Our Titanic data satisfies this

4. **Approximately linear relationship between continuous predictors and the log-odds** — this is the key "linear" assumption. It's often reasonable in practice but can be checked with residual plots

5. **Adequate sample size** — logistic regression needs enough samples per feature to produce stable estimates. A rule of thumb is at least 10 events per predictor variable

In practice, logistic regression is quite robust to mild violations of these assumptions. The biggest practical concern is **feature scaling** — as we discussed, always scale your features for logistic regression.

## Summary

In this notebook, you learned:

1. **Logistic regression** predicts the probability of a binary outcome using the sigmoid function
2. **Feature engineering** — encoding categorical variables, log-transforming skewed features
3. **Stratified train-test splits** preserve class ratios
4. **Pipelines** chain preprocessing and modeling to prevent data leakage
5. **Feature scaling** is essential for logistic regression (but not tree models)
6. **Odds ratios** make coefficients interpretable
7. **Baselines** help us judge whether a model actually adds value
8. **Classification reports** give per-class precision, recall, and F1
9. **ROC curves and AUC** measure ranking ability across all thresholds
10. **Precision-Recall curves** provide deeper insight into imbalanced data performance
11. **Cross-validation** ensures model stability and robustness
12. **Threshold tuning** lets us trade off precision and recall based on business context

---

## Problems

### Problem 1
Explain precision, recall, and F1 score. Be clear and precise about what each one measures.

Write your answer here.

### Problem 2
Look at the classification report for our model. Explain what the precision, recall, and F1 values mean in terms of our model's performance on the Titanic survival prediction task. Which class does the model handle better and why?

Write your answer here.

### Problem 3
Find the best logistic regression model using only **four features** from the original set. You can use any approach: exhaustive search over feature combinations, L1 regularization, or forward/backward selection.

**Hint:** If you want to try L1 regularization, set `penalty='l1'` and `solver='liblinear'` in your LogisticRegression. The `C` parameter controls regularization strength (smaller C = more regularization, more features pushed to zero).

**Hint for exhaustive search:** The `itertools.combinations` module can generate all 4-feature subsets. Loop through them and track the best test accuracy.

In [ ]:
# Enter your code here
# Try finding the best 4-feature logistic regression model




### Problem 4
Explain your solution to Problem 3. What approach did you use? What was the best 4-feature set you found, and how did you determine it was the best? How does the 4-feature model's performance compare to the full 7-feature model?

Write your answer here.